# Vesuvius Scroll Data Visualizer
Explore the 7 downloaded scroll segments: Z-stack slices, masks, depth profiles, and patch sampling.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.widgets import Slider
import tifffile
from PIL import Image
import json
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

DATA_DIR = Path('../data/scroll')
SEGMENTS = sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()])
print(f'Found {len(SEGMENTS)} segments:')
for s in SEGMENTS:
    tifs = list((DATA_DIR / s / 'surface_volume').glob('*.tif'))
    size_gb = sum(t.stat().st_size for t in tifs) / 1e9
    print(f'  {s}  —  {len(tifs)} layers  —  {size_gb:.2f} GB')

## 1. Segment Overview — Middle Slice + Mask

In [ ]:
def load_slice(segment_id, z=32):
    path = DATA_DIR / segment_id / 'surface_volume' / f'{z:02d}.tif'
    return tifffile.imread(str(path)).astype(np.float32)

def load_mask(segment_id):
    path = DATA_DIR / segment_id / 'mask.png'
    return np.array(Image.open(str(path)).convert('L')) > 0

fig, axes = plt.subplots(2, len(SEGMENTS), figsize=(4 * len(SEGMENTS), 8))
fig.suptitle('All Segments — Middle Slice (z=32) and Mask', fontsize=14, y=1.01)

for i, seg in enumerate(SEGMENTS):
    sl = load_slice(seg, z=32)
    mask = load_mask(seg)

    # normalize for display
    p1, p99 = np.percentile(sl[mask], [1, 99])
    sl_disp = np.clip((sl - p1) / (p99 - p1 + 1e-8), 0, 1)

    axes[0, i].imshow(sl_disp, cmap='gray')
    axes[0, i].set_title(seg[:8] + '\n' + seg[8:], fontsize=7)
    axes[0, i].axis('off')

    axes[1, i].imshow(mask, cmap='gray')
    axes[1, i].set_title('mask', fontsize=7)
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

## 2. Interactive Z-Stack Browser — Scrub Through All 65 Layers

In [ ]:
seg_dropdown = widgets.Dropdown(options=SEGMENTS, description='Segment:')
z_slider = widgets.IntSlider(value=32, min=0, max=64, step=1, description='Z layer:', continuous_update=True)
contrast_toggle = widgets.Checkbox(value=True, description='Auto contrast')

out = widgets.Output()

def update(change):
    with out:
        out.clear_output(wait=True)
        seg = seg_dropdown.value
        z = z_slider.value
        sl = load_slice(seg, z)
        mask = load_mask(seg)

        fig, axes = plt.subplots(1, 3, figsize=(16, 5))
        fig.suptitle(f'Segment {seg}  —  Layer z={z:02d}', fontsize=12)

        if contrast_toggle.value:
            p1, p99 = np.percentile(sl[mask], [1, 99])
            sl_disp = np.clip((sl - p1) / (p99 - p1 + 1e-8), 0, 1)
        else:
            sl_disp = sl / 65535.0

        # raw slice
        axes[0].imshow(sl_disp, cmap='gray')
        axes[0].set_title('CT slice')
        axes[0].axis('off')

        # slice masked
        masked = sl_disp.copy()
        masked[~mask] = 0
        axes[1].imshow(masked, cmap='gray')
        axes[1].set_title('Masked (valid papyrus only)')
        axes[1].axis('off')

        # histogram of intensities inside mask
        axes[2].hist(sl[mask].ravel(), bins=200, color='steelblue', alpha=0.8)
        axes[2].set_title('Intensity histogram (masked region)')
        axes[2].set_xlabel('Raw pixel value (16-bit)')
        axes[2].set_ylabel('Count')

        plt.tight_layout()
        plt.show()

seg_dropdown.observe(update, names='value')
z_slider.observe(update, names='value')
contrast_toggle.observe(update, names='value')

display(widgets.HBox([seg_dropdown, z_slider, contrast_toggle]), out)
update(None)

## 3. Depth Profile — How Intensity Changes Across All 65 Z-Layers
Shows how CT density varies from above the papyrus surface (z=0) through to below it (z=64). Ink, if present, would show as local density spikes near the surface layers.

In [ ]:
def compute_depth_profile(segment_id):
    mask = load_mask(segment_id)
    means, stds = [], []
    for z in range(65):
        sl = load_slice(segment_id, z)
        vals = sl[mask]
        means.append(vals.mean())
        stds.append(vals.std())
    return np.array(means), np.array(stds)

fig, ax = plt.subplots(figsize=(14, 5))
colors = plt.cm.tab10(np.linspace(0, 1, len(SEGMENTS)))

for seg, color in zip(SEGMENTS, colors):
    means, stds = compute_depth_profile(seg)
    z = np.arange(65)
    ax.plot(z, means, label=seg, color=color, linewidth=1.5)
    ax.fill_between(z, means - stds, means + stds, alpha=0.1, color=color)

ax.axvline(32, color='black', linestyle='--', alpha=0.4, label='z=32 (surface)')
ax.set_xlabel('Z layer (0=above surface, 32=surface, 64=below)')
ax.set_ylabel('Mean CT intensity (masked pixels)')
ax.set_title('Depth Profile — All Segments (shaded = ±1 std dev)')
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

## 4. Z-Stack as Filmstrip — All 65 Layers for One Segment

In [ ]:
FILMSTRIP_SEGMENT = SEGMENTS[-1]  # change this to any segment ID

# downsample each slice for display
THUMB_SIZE = 128

fig, axes = plt.subplots(5, 13, figsize=(26, 10))
fig.suptitle(f'All 65 Z-layers — Segment {FILMSTRIP_SEGMENT}', fontsize=12)
axes = axes.ravel()

mask = load_mask(FILMSTRIP_SEGMENT)

for z in range(65):
    sl = load_slice(FILMSTRIP_SEGMENT, z)
    p1, p99 = np.percentile(sl[mask], [1, 99])
    sl_disp = np.clip((sl - p1) / (p99 - p1 + 1e-8), 0, 1)
    # thumbnail
    thumb = np.array(Image.fromarray((sl_disp * 255).astype(np.uint8)).resize((THUMB_SIZE, THUMB_SIZE)))
    axes[z].imshow(thumb, cmap='gray')
    axes[z].set_title(f'z={z:02d}', fontsize=6)
    axes[z].axis('off')

for ax in axes[65:]:
    ax.axis('off')

plt.tight_layout()
plt.show()

## 5. Sample Patches — What the Model Will See
Extracts random 224×224 patches from inside the valid mask region, shown as a 4×4 grid.

In [ ]:
PATCH_SEGMENT = SEGMENTS[-1]  # Grand Prize segment
PATCH_SIZE = 224
N_PATCHES = 16
Z_SURFACE = 32

np.random.seed(42)
mask = load_mask(PATCH_SEGMENT)
sl = load_slice(PATCH_SEGMENT, Z_SURFACE)

H, W = sl.shape
valid_ys, valid_xs = np.where(
    mask[PATCH_SIZE//2 : H - PATCH_SIZE//2, PATCH_SIZE//2 : W - PATCH_SIZE//2]
)

chosen = np.random.choice(len(valid_ys), size=N_PATCHES, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
fig.suptitle(f'Random 224×224 Patches — Segment {PATCH_SEGMENT}  (z=32, surface layer)', fontsize=11)

for i, idx in enumerate(chosen):
    cy = valid_ys[idx] + PATCH_SIZE // 2
    cx = valid_xs[idx] + PATCH_SIZE // 2
    patch = sl[cy - PATCH_SIZE//2 : cy + PATCH_SIZE//2,
               cx - PATCH_SIZE//2 : cx + PATCH_SIZE//2]
    p1, p99 = np.percentile(patch, [1, 99])
    patch_disp = np.clip((patch - p1) / (p99 - p1 + 1e-8), 0, 1)
    axes[i // 4, i % 4].imshow(patch_disp, cmap='gray')
    axes[i // 4, i % 4].set_title(f'({cx},{cy})', fontsize=7)
    axes[i // 4, i % 4].axis('off')

plt.tight_layout()
plt.show()

## 6. Cross-Section View — Vertical Slice Through Z-Stack
Cuts vertically through the Z-stack to show the papyrus sheet in cross-section. The bright band is the papyrus surface.

In [ ]:
CROSS_SEGMENT = SEGMENTS[-1]

mask = load_mask(CROSS_SEGMENT)

# pick a row near the center of the valid region
valid_rows = np.where(mask.any(axis=1))[0]
row = valid_rows[len(valid_rows) // 2]

# stack all Z layers at that row → shape (65, W)
cross = np.stack([load_slice(CROSS_SEGMENT, z)[row] for z in range(65)], axis=0)

p1, p99 = np.percentile(cross, [1, 99])
cross_disp = np.clip((cross - p1) / (p99 - p1 + 1e-8), 0, 1)

fig, ax = plt.subplots(figsize=(16, 4))
ax.imshow(cross_disp, cmap='gray', aspect='auto')
ax.axhline(32, color='red', linestyle='--', linewidth=0.8, label='z=32 surface')
ax.set_xlabel('X position (pixels)')
ax.set_ylabel('Z layer (0=top, 64=bottom)')
ax.set_title(f'Cross-section through Z-stack at row {row} — Segment {CROSS_SEGMENT}')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Segment Statistics Summary

In [ ]:
print(f'{"Segment":<20} {"Shape (H×W)":<18} {"Valid px":<12} {"Coverage":<10} {"Mean (z32)":<12} {"Std (z32)"}')
print('-' * 85)

for seg in SEGMENTS:
    sl = load_slice(seg, 32)
    mask = load_mask(seg)
    H, W = sl.shape
    valid = mask.sum()
    coverage = valid / (H * W) * 100
    mean_val = sl[mask].mean()
    std_val = sl[mask].std()
    print(f'{seg:<20} {H}×{W:<12} {valid:<12,} {coverage:<9.1f}% {mean_val:<12.0f} {std_val:.0f}')